In [1]:
import math
import time
import numpy as np
import torch


def calculate_fps(prev_time, fps_buffer, buffer_size):
    current_time = time.time()
    time_diff = current_time - prev_time
    if time_diff == 0:
        fps = 0
    else:
        fps = 1.0 / time_diff
    fps_buffer.append(fps)
    if len(fps_buffer) > buffer_size:
        fps_buffer.pop(0)
    avg_fps = sum(fps_buffer) / len(fps_buffer) if fps_buffer else 0
    return avg_fps, current_time



# Функція для експоненційного згладжування координат з перевіркою на від'ємні значення та вихід за межі кадру
def smooth_coordinates(current_coords, prev_coords, frame_width, frame_height, alpha=0.2):
    smoothed_coords = []
    for cur, prev in zip(current_coords, prev_coords):
        smoothed_value = alpha * cur + (1 - alpha) * prev
        
        # Перевірка на від'ємні значення та вихід за межі кадру
        if smoothed_value < 0:
            smoothed_value = 0
        elif smoothed_value > frame_width and (len(smoothed_coords) % 2 == 0):  # Якщо координата x
            smoothed_value = frame_width
        elif smoothed_value > frame_height and (len(smoothed_coords) % 2 == 1):  # Якщо координата y
            smoothed_value = frame_height
        
        smoothed_coords.append(smoothed_value)
    
    return smoothed_coords


def compute_score(x1, y1, x2, y2, center_screen_x, center_screen_y):
    # Обчислюємо площу рамки
    area = (x2 - x1) * (y2 - y1)
    
    # Обчислюємо центр рамки
    center_box_x = (x1 + x2) / 2
    center_box_y = (y1 + y2) / 2
    
    # Обчислюємо відстань від центру рамки до центру екрану
    distance = math.sqrt((center_box_x - center_screen_x) ** 2 + (center_box_y - center_screen_y) ** 2)
    
    # Об'єднуємо площу та відстань для визначення "найбільшої та найближчої" рамки.
    # Використовуємо відстань як вагу, щоб обрати найближчу, і площу для вибору найбільшої рамки.
    score = 2 * distance / area  # Чим менша відстань і більша площа, тим краща рамка
    return score


def select_best_box_and_platenumber(detections, center_screen_x, center_screen_y):
    # Змінні для зберігання найкращої рамки автомобіля та пов'язаного score
    best_car_box = None
    best_score = float('inf')  # Чим менше значення, тим краща рамка

    # Змінні для зберігання номерного знака всередині найкращого автомобіля
    licence_number_box = None

    licence_score = 0
    best_car_box_score = 0

    # Спочатку обробляємо детекції, щоб знайти найкращу рамку автомобіля
    for detection in detections.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = detection

        if class_id == 1:  # Клас 1 - автомобіль
            box_score = compute_score(x1, y1, x2, y2, center_screen_x, center_screen_y)
            
            # Якщо нова рамка має кращий "score", зберігаємо її як найкращу рамку автомобіля
            if box_score < best_score:
                best_score = box_score
                best_car_box = [x1, y1, x2, y2]
                best_car_box_score = score

    # Якщо ми знайшли найкращу рамку автомобіля, перевіряємо наявність номерного знака всередині неї
    if best_car_box is not None:
        car_x1, car_y1, car_x2, car_y2 = best_car_box

        # Змінна для зберігання найкращого номерного знака (з найвищим score)
        licence_number_box = None

        for detection in detections.boxes.data.tolist():
            x1, y1, x2, y2, score, class_id = detection

            if class_id == 0:  # Клас 0 - номерний знак
                # Перевіряємо, чи рамка номерного знака знаходиться всередині рамки автомобіля
                if (x1 >= car_x1 and y1 >= car_y1 and x2 <= car_x2 and y2 <= car_y2):
                    # Якщо знайдено номерний знак з кращим score, зберігаємо його
                    licence_number_box = [x1, y1, x2, y2]
                    licence_score = score

    # Повертаємо найкращу рамку автомобіля та рамку номерного знака (якщо знайдено)
    return best_car_box, best_car_box_score, licence_number_box, licence_score

def select_best_box_and_platenumber2(detections, center_screen_x, center_screen_y):
    """
    Повертає (best_car_box, best_car_box_score, licence_number_box, licence_score)
    
    Логіка вибору автомобіля (class_id=1):
    1) Беремо тільки ті рамки, що містять центр кадру.
    2) Із них вибираємо рамку з мінімальною відстанню від центру рамки до центру кадру.
    3) Якщо нема жодної рамки, що містить центр, повертаємо (None, 0, None, 0).
    
    Логіка вибору номерного знака (class_id=0):
    - Серед рамок, що повністю лежать у вибраній рамці авто, вибираємо ту, що має найбільший score.
    """

    best_car_box = None
    best_car_box_score = 0.0
    best_distance = float("inf")
    
    # =============================
    # 1. Знаходимо авто, що містить центр кадру
    # =============================
    for detection in detections.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = detection

        if int(class_id) == 1:  # Клас 1 - автомобіль
            # Перевіряємо, чи рамка містить центр кадру
            if x1 <= center_screen_x <= x2 and y1 <= center_screen_y <= y2:
                # Центр рамки
                box_center_x = (x1 + x2) / 2
                box_center_y = (y1 + y2) / 2

                # Відстань від центру цієї рамки до центру кадру
                distance = math.hypot(box_center_x - center_screen_x,
                                      box_center_y - center_screen_y)

                # Якщо ця відстань менша за поточну найкращу – оновлюємо
                if distance < best_distance:
                    best_distance = distance
                    best_car_box = [x1, y1, x2, y2]
                    best_car_box_score = score  # Запам'ятовуємо впевненість детектора

    # Якщо не знайшли жодної рамки, що містить центр, повертаємо None
    if best_car_box is None:
        return None, 0.0, None, 0.0

    # =============================
    # 2. Шукаємо номерний знак (class_id=0) всередині вибраного авто
    # =============================
    licence_number_box = None
    licence_score = 0.0

    car_x1, car_y1, car_x2, car_y2 = best_car_box
    for detection in detections.boxes.data.tolist():
        x1, y1, x2, y2, score, class_id = detection

        if int(class_id) == 0:  # Клас 0 – номерний знак
            # Перевіряємо, чи рамка номерного знака міститься в авто
            if x1 >= car_x1 and y1 >= car_y1 and x2 <= car_x2 and y2 <= car_y2:
                # Обираємо той, що має найбільший score
                if score > licence_score:
                    licence_score = score
                    licence_number_box = [x1, y1, x2, y2]

    return best_car_box, best_car_box_score, licence_number_box, licence_score

In [3]:
import cv2
import numpy as np
import onnx
import onnxruntime
from tracking.video_demo import HiT
from yolov10.ultralytics import YOLOv10

import PIL
PIL.Image.ANTIALIAS = PIL.Image.LANCZOS

import torch
torch.cuda.is_available()

c:\Users\krapa\anaconda3\envs\licence_plate_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\krapa\anaconda3\envs\licence_plate_project\lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
c:\Users\krapa\anaconda3\envs\licence_plate_project\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


True

In [4]:
import torch

if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
else:
    print("CUDA is NOT available. Please check your setup.")


CUDA is available!
Device name: NVIDIA GeForce RTX 4070 SUPER
CUDA version: 12.4
cuDNN version: 90100


In [4]:
# from fast.fast_plate_ocr import ONNXPlateRecognizer

# m = ONNXPlateRecognizer('european-plates-mobile-vit-v2-model')
# print(m.run(r"C:\Users\krapa\Downloads\autoriaNumberplateOcrUa-2023-04-18\val\img\21054784-21054784--BA5599AX-BMW_5_Series_0.png"))

In [5]:
import cv2
import threading
import queue
import time
from fast.fast_plate_ocr import ONNXPlateRecognizer

m = ONNXPlateRecognizer('european-plates-mobile-vit-v2-model')


net_path = 'C:/Work_copy/HiT/HiT_Tiny/VT_ep1500.onnx'
onnx_model = onnx.load(net_path)
onnx.checker.check_model(onnx_model)
with torch.no_grad():
    ort_session = onnxruntime.InferenceSession(net_path, providers=['CUDAExecutionProvider'])
    print(onnxruntime.get_device())


*************** EP Error ***************
EP Error D:\a\_work\1\s\onnxruntime\python\onnxruntime_pybind_state.cc:507 onnxruntime::python::RegisterTensorRTPluginsAsCustomOps Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************
GPU


In [7]:
import torch
print(torch.cuda.is_available())  # Доступність CUDA
print(torch.cuda.device_count())  # Кількість доступних GPU
print(torch.cuda.get_device_name(0))  # Назва першого GPU (якщо є)


True
1
NVIDIA GeForce RTX 4070 SUPER


In [1]:
import onnxruntime
print(onnxruntime.get_device())  # Очікуваний вивід: 'GPU'


GPU


In [5]:
import cv2
import threading
import queue
import time
from fast.fast_plate_ocr import ONNXPlateRecognizer

m = ONNXPlateRecognizer('european-plates-mobile-vit-v2-model')

# Ініціалізація моделей та трекерів
coco_model = YOLOv10("C:/Work_copy/licence_plate_project/runs/detect/train15/weights/best.pt")

net_path = 'C:/Work_copy/HiT/HiT_Tiny/VT_ep1500.onnx'
onnx_model = onnx.load(net_path)
onnx.checker.check_model(onnx_model)
with torch.no_grad():
    ort_session = onnxruntime.InferenceSession(net_path, providers=['CUDAExecutionProvider'])
    print(onnxruntime.get_device())

# Трекери
tracker = HiT(ort_session)
license_tracker = HiT(ort_session)

# Шлях до відео
video_path = 'Video_v2.mp4'

# Черги для кадрів
frame_queue = queue.Queue(maxsize=10)         # Для захоплених кадрів
processed_queue = queue.Queue(maxsize=10)     # Для оброблених кадрів

# Змінні для контролю потоків
car_detected = False
licence_detected = False
stop_threads = False

# Отримуємо розміри кадру для обчислення центру
# Читаємо перший кадр поза потоками
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
if not ret:
    print("Не вдалося отримати кадр з відео.")
    exit()
height, width = frame.shape[:2]
center_screen_x = width / 2
center_screen_y = height / 2
cap.release()  # Звільняємо ресурс, оскільки він буде створений у потоці

# Функція для захоплення кадрів
def capture_frames():
    cap = cv2.VideoCapture(video_path)  # Створюємо об'єкт cap всередині потоку
    while not stop_threads:
        ret, frame = cap.read()
        if not ret:
            break
        frame_queue.put(frame)
    frame_queue.put(None)  # Сигнал про завершення
    cap.release()

# Функція для обробки кадрів
def process_frames():
    global car_detected, licence_detected  # Додаємо, якщо змінюємо ці змінні
    frame_nmr = -1
    prev_time = time.time()
    fps_buffer = []
    buffer_size = 60

    previous_coords = {'car': None, 'licence': None}

    while True:
        frame = frame_queue.get()
        if frame is None:
            break
        frame_nmr += 1

        avg_fps, prev_time = calculate_fps(prev_time, fps_buffer, buffer_size)


        # Копіюємо кадр для відображення
        display_frame = frame.copy()

        cv2.putText(display_frame, f'FPS: {avg_fps:.0f}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        if frame_nmr % 15 == 0:
            # Детекція
            detections = coco_model(frame)[0]
            
            if detections:
                car_box, car_score, licencenumber_box, licencenumber_score = select_best_box_and_platenumber2(
                    detections, center_screen_x, center_screen_y)
                
                print(car_score, licencenumber_score)
                
                if car_box and car_score > 0.7:
                    c_x1, c_y1, c_x2, c_y2 = [int(s) for s in car_box]
                    tracker.initialize(frame, {'init_bbox': [c_x1, c_y1, c_x2 - c_x1, c_y2 - c_y1]})
                    car_detected = True
                else:
                    car_detected = False
                
                if licencenumber_box and licencenumber_score > 0.7:
                    l_x1, l_y1, l_x2, l_y2 = [int(s) for s in licencenumber_box]
                    license_tracker.initialize(frame, {'init_bbox': [l_x1, l_y1, l_x2 - l_x1, l_y2 - l_y1]})

                    licence_plate, conf = m.run(cv2.cvtColor(frame[l_y1:l_y2, l_x1:l_x2], cv2.COLOR_BGR2GRAY), return_confidence=True)

                    licence_plate = licence_plate[0].replace('_','')
                    conf = conf[0][:len(licence_plate)]
                    if np.all(conf > 0.5):
                        licence_detected = True
                    else:
                        licence_detected = False
                else:
                    licence_detected = False

            else:
                car_detected = False
                licence_detected = False
        else:
            if car_detected:
                car_out = tracker.track(frame)
                state = [int(s) for s in car_out['target_bbox']]
                c_x1, c_y1, w, h = state
                c_x2, c_y2 = c_x1 + w, c_y1 + h
            
            if licence_detected:
                licence_out = license_tracker.track(frame)
                state = [int(s) for s in licence_out['target_bbox']]
                l_x1, l_y1, w, h = state
                l_x2, l_y2 = l_x1 + w, l_y1 + h
        

        if car_detected:
            if previous_coords['car']:
                previous_coords['car'] = smooth_coordinates([c_x1, c_y1, c_x2, c_y2], previous_coords['car'], width, height, alpha=0.2)
            else:
                previous_coords['car'] = [c_x1, c_y1, c_x2, c_y2]

            c_x1, c_y1, c_x2, c_y2 = [int(coord) for coord in previous_coords['car']]
            cv2.rectangle(display_frame, (c_x1, c_y1), (c_x2, c_y2), (0, 0, 255), 5)
            
        if licence_detected:
            if previous_coords['licence']:
                previous_coords['licence'] = smooth_coordinates([l_x1, l_y1, l_x2, l_y2], previous_coords['licence'], width, height, alpha=0.7)
            else:
                previous_coords['licence'] = [l_x1, l_y1, l_x2, l_y2]

            l_x1, l_y1, l_x2, l_y2 = [int(coord) for coord in previous_coords['licence']]
            cv2.rectangle(display_frame, (l_x1, l_y1), (l_x2, l_y2), (255, 0, 0), 5)
            

            text_size = cv2.getTextSize(f'{licence_plate}', cv2.FONT_HERSHEY_SIMPLEX, 1, 3)[0]


            cv2.rectangle(display_frame, (l_x1 - 15, l_y1), (l_x1 - 15 + text_size[0] + 10, l_y1 - 15 - text_size[1] - 10), (0, 0, 0), -1)
            cv2.putText(display_frame, f'{licence_plate}', (l_x1-15, l_y1-15), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)

        
        processed_queue.put(display_frame)
    processed_queue.put(None)  # Сигнал про завершення

# Функція для відображення кадрів
def display_frames():
    while True:
        frame = processed_queue.get()
        if frame is None:
            break
        cv2.imshow('Result', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            global stop_threads
            stop_threads = True
            break
    cv2.destroyAllWindows()

# Запускаємо потоки
capture_thread = threading.Thread(target=capture_frames)
process_thread = threading.Thread(target=process_frames)
display_thread = threading.Thread(target=display_frames)

capture_thread.start()
process_thread.start()
display_thread.start()

# Очікуємо завершення потоків
capture_thread.join()
process_thread.join()
display_thread.join()

*************** EP Error ***************
EP Error D:\a\_work\1\s\onnxruntime\python\onnxruntime_pybind_state.cc:490 onnxruntime::python::RegisterTensorRTPluginsAsCustomOps Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************


c:\Users\krapa\anaconda3\envs\licence_plate_project\lib\site-packages\ultralytics\nn\tasks.py:733: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_

GPU

0: 384x640 1 licence, 5 cars, 39.0ms
Speed: 3.2ms preprocess, 39.0ms inference, 46.6ms postprocess per image at shape (1, 3, 384, 640)
0.0 0.0

0: 384x640 1 licence, 6 cars, 12.3ms
Speed: 2.0ms preprocess, 12.3ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
0.0 0.0

0: 384x640 1 licence, 4 cars, 9.5ms
Speed: 2.0ms preprocess, 9.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
0.0 0.0

0: 384x640 1 licence, 5 cars, 15.0ms
Speed: 1.0ms preprocess, 15.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
0.9194604158401489 0.0

0: 384x640 1 licence, 3 cars, 10.0ms
Speed: 1.0ms preprocess, 10.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
0.9153497815132141 0.0

0: 384x640 3 licences, 3 cars, 9.1ms
Speed: 1.0ms preprocess, 9.1ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)
0.9321173429489136 0.0

0: 384x640 2 licences, 4 cars, 9.0ms
Speed: 1.0ms preprocess, 9.0ms inference, 0.0ms postproce